In [1]:
import torch
import onnx
from onnxsim import simplify
import copy
import numpy as np
import random
from torch.onnx import register_custom_op_symbolic
from torch.onnx.symbolic_helper import parse_args

import sys 
#sys.path.append(".")
# Path Definitions

pth_path = '/home/dongz/FR-UNet_dysample/saved/FR_UNet_Quan_BiLinear/DRIVE/250306113203/checkpoint-epoch32.pth'
onnx_path = '/home/dongz/FR-UNet_dysample/outputs/onnx/frunet_fp_quan_192p.onnx'
fxp_onnx_path = '/home/dongz/FR-UNet_dysample/outputs/onnx/frunet_quan_192p_latest_2.onnx'
npz_path = '/home/dongz/FR-UNet_dysample/npz_logging/'

/home/dongz/anaconda3/envs/fr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import shutil
import os

def clean_directory(directory):
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)

clean_directory(npz_path)


In [3]:
import models
from ruamel.yaml import YAML
from bunch import Bunch
from utils.helpers import get_instance

yaml = YAML(typ='safe', pure=True)
with open('config_quan.yaml', 'r') as file:
    CFG = Bunch(yaml.load(file))

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    print("CUDA is not available.")
    device = torch.device('cpu')
    
model = get_instance(models, 'model', CFG).to(device)

# 加载模型的状态
pretrained_dict = torch.load(pth_path, map_location=device)

# 移除因为分布式训练而有的 'module.' 前缀
new_state_dict = {}
for k, v in pretrained_dict.items():
    if k.startswith('module.'):
        k = k[7:]  # 移除 'module.' 前缀
    new_state_dict[k] = v

# 加载修改后的状态字典到目标模型
model.load_state_dict(new_state_dict, strict=False)
model.eval()


FR_UNet_Quan(
  (block1_3): block(
    (fuse): feature_fuse(
      (conv11): QuanConv(
        1, 32, kernel_size=(1, 1), stride=(1, 1), bias=False
        (lsq_w): LsqQuantizer4weight()
        (quan_w): LsqQuantizer4weight()
        (lsq_a): LsqQuantizer4input()
        (quan_a): LsqQuantizer4input()
      )
      (conv33): QuanConv(
        1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (lsq_w): LsqQuantizer4weight()
        (quan_w): LsqQuantizer4weight()
        (lsq_a): LsqQuantizer4input()
        (quan_a): LsqQuantizer4input()
      )
      (conv33_di): QuanConv(
        1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), dilation=(2, 2), bias=False
        (lsq_w): LsqQuantizer4weight()
        (quan_w): LsqQuantizer4weight()
        (lsq_a): LsqQuantizer4input()
        (quan_a): LsqQuantizer4input()
      )
      (quan_): ModuleDict(
        (fuse_1_16bit): LsqQuantizer4input()
        (fuse_2_16bit): LsqQuantizer4input()
        (fuse_1_8

In [4]:
dummy_input = torch.rand(1, 1, 192, 192).to(device)

# 进行推理
with torch.no_grad():
    dummy_output = model(dummy_input)

lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_input begin
lsq init_inpu

#### Log Scaling Factors

#### Start FXP ONNX Conversion

In [5]:
model = onnx.load(onnx_path)
npz_idx = 0
output_map = dict()
input_map = dict()

##### Auxiliary Functions

In [6]:
def has_attribute(attr_list, name):
    has_attr = False
    for a in attr_list:
        if a.name == name:
            has_attr = True
            break
    return has_attr
def get_attribute(attr_list, name):
    for a in attr_list:
        if a.name == name:
            return a
    assert False
def set_output_scale (node, output_scale):
    for i in node.input:
        if i in output_map: # 如果outputmap中有这个input，即和当前node相连的上一个node
            for d in output_map[i]: # 遍历这个input的所有上一个node
                if not has_attribute(d.attribute, "output_scale"): # 如果上一个node没有output_scale
                    attr = onnx.helper.make_attribute("output_bitdepth", 8) # 添加output_bitdepth
                    d.attribute.append(attr)
                    attr = onnx.helper.make_attribute("output_scale", output_scale) # 添加output_scale
                    d.attribute.append(attr) 
                    set_output_scale (d, output_scale) # 向上追溯没有output_scale的node,即输入输出scale相同的node
# Initialization
for node in model.graph.node:
    node.doc_string = ""
    if node.output[0] not in output_map:
        output_map[node.output[0]] = []
    output_map[node.output[0]].append(node)
    for i in node.input:
        if i not in input_map:
            input_map[i] = []
        input_map[i].append(node)
        
for i, node in enumerate(model.graph.node):
    print(f"Node {i}: {node.op_type}")

Node 0: Conv
Node 1: Conv
Node 2: Conv
Node 3: Add
Node 4: Add
Node 5: Conv
Node 6: HardSwish
Node 7: Conv
Node 8: HardSwish
Node 9: Conv
Node 10: HardSwish
Node 11: Conv
Node 12: HardSwish
Node 13: Conv
Node 14: HardSwish
Node 15: Conv
Node 16: HardSwish
Node 17: Conv
Node 18: HardSwish
Node 19: Conv
Node 20: HardSwish
Node 21: Conv
Node 22: HardSwish
Node 23: Resize
Node 24: Conv
Node 25: HardSwish
Node 26: Concat
Node 27: Conv
Node 28: Conv
Node 29: Conv
Node 30: Add
Node 31: Add
Node 32: Conv
Node 33: HardSwish
Node 34: Conv
Node 35: HardSwish
Node 36: Conv
Node 37: HardSwish
Node 38: Concat
Node 39: Conv
Node 40: Conv
Node 41: Conv
Node 42: Add
Node 43: Add
Node 44: Conv
Node 45: HardSwish
Node 46: Conv
Node 47: HardSwish
Node 48: Conv
Node 49: HardSwish
Node 50: Resize
Node 51: Conv
Node 52: HardSwish
Node 53: Conv
Node 54: HardSwish
Node 55: Conv
Node 56: HardSwish
Node 57: Conv
Node 58: HardSwish
Node 59: Resize
Node 60: Conv
Node 61: HardSwish
Node 62: Concat
Node 63: Conv
Nod

In [ ]:
for node in model.graph.node:
    if node.op_type == "Conv":
        npz_idx = npz_idx + 1
        val = np.load(npz_path + "/" + str(npz_idx) + "_conv.npz")
        input_scale = val["input_scale"]
        d = node
        if d.input[0] in output_map: # 如果outputmap中有这个input，即和当前node相连的上一个node
            if has_attribute(output_map[d.input[0]][0].attribute, "output_scale"): # 如果上一个node有output_scale
                input_node_out_scale = get_attribute(output_map[d.input[0]][0].attribute, "output_scale").f # 获取上一个node的output_scale
                input_scale[0] = 1/ input_node_out_scale # 计算当前node的input_scale
                assert input_node_out_scale == 1/input_scale[0], str(input_node_out_scale) + "-" + str(1/input_scale[0]) # 检查是否相等
            # 如果上一个node没有output_scale，那么就是输入的scale，那么当前node的input_scale就是1/输入的scale
            set_output_scale (node, 1 / input_scale[0]) # 设置当前node的所有上一个node的output_scale
        # 这里设置上一个node的input_scale 
        if node.input[0] in output_map:
            for d in output_map[node.input[0]]:
                if not has_attribute(d.attribute, "output_scale"):
                    attr = onnx.helper.make_attribute("output_bitdepth", 8)
                    d.attribute.append(attr)
                    attr = onnx.helper.make_attribute("output_scale", 1/input_scale[0])
                    d.attribute.append(attr)  
                
        w = val["w"]
        weight_name = node.input[1]
        ## edit weights
        for idx, t in enumerate(model.graph.initializer):
            if t.name == weight_name:
                value = onnx.helper.make_tensor(t.name, onnx.TensorProto.FLOAT, t.dims, w.flatten())
                model.graph.initializer.remove(t)
                model.graph.initializer.insert(idx, value)
                break
        if "b" in val:
            b = val["b"]
            bias_name = node.input[2]
            value = onnx.helper.make_tensor("bias_" + str(npz_idx), onnx.TensorProto.FLOAT, b.shape, b.flatten())
            model.graph.initializer.append(value)
            node.input[2] = "bias_" + str(npz_idx)
            # ## edit weights
            # for idx, t in enumerate(model.graph.initializer):
            #     if t.name == bias_name:
            #         print (bias_name, t.name, t.dims, b.flatten())
            #         input()
            #         value = onnx.helper.make_tensor(t.name, onnx.TensorProto.FLOAT, t.dims, b.flatten())
            #         model.graph.initializer.remove(t)
            #         model.graph.initializer.insert(idx, value)
            #         break
                
        # del node.attribute[:]
        attr = onnx.helper.make_attribute("input_bitdepth", 8)
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("input_scale", 1/input_scale[0])
        node.attribute.append(attr)   
        attr = onnx.helper.make_attribute("weight_bitdepth", 8)
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("weight_ch_scales", (1 / val["weight_scales"]).flatten())
        node.attribute.append(attr)   
        attr = onnx.helper.make_attribute("bias_bitdepth", 16)
        node.attribute.append(attr)
        
    elif node.op_type == "HardSwish":
        npz_idx = npz_idx + 1
        val = np.load(npz_path + "/" + str(npz_idx) + "_pwl.npz")
        
        input_scale = 1/val["input_scale"][0]
        attr = onnx.helper.make_attribute("slopes", val["slopes"])
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("intercepts", val["intercepts"])
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("slopes_scale", 1/val["slopes_scale"][0])
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("intercepts_scale", 1/val["intercepts_scale"][0])
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("change_pts", val["change_pts"])
        node.attribute.append(attr)
        
        if node.input[0] in output_map:
            for d in output_map[node.input[0]]:
                if not has_attribute(d.attribute, "output_scale"):
                    attr = onnx.helper.make_attribute("output_bitdepth", 8)
                    d.attribute.append(attr)
                    attr = onnx.helper.make_attribute("output_scale", 1/val["input_scale"][0])
                    d.attribute.append(attr)  
        # del node.attribute[:]
        attr = onnx.helper.make_attribute("input_bitdepth", 8)
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("input_scale", 1/val["input_scale"][0])
        node.attribute.append(attr)  
         
    elif node.op_type == "Add":
        npz_idx = npz_idx + 1
        try:
            val = np.load(npz_path + "/" + str(npz_idx) +  "_add.npz")
        except Exception as e:
            print(f"Error occurred: {e}")  # 打印异常信息
            npz_idx = npz_idx - 1   
            continue
            
        for input1 in node.input:
            if input1 in output_map:
                for d in output_map[input1]:
                    if not has_attribute(d.attribute, "output_scale"):
                        if d.op_type == "Conv":
                            attr = onnx.helper.make_attribute("output_bitdepth", 8) #16
                        else:
                            attr = onnx.helper.make_attribute("output_bitdepth", 8)
                        d.attribute.append(attr)
                        attr = onnx.helper.make_attribute("output_scale", 1/val["output_scale"][0])
                        d.attribute.append(attr) 
        # new added 
        attr = onnx.helper.make_attribute("output_bitdepth", 8)
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("output_scale", 1/val["output_scale"][0])
        node.attribute.append(attr) 

    elif node.op_type == "MatMul":
        # check if input 1 is a constant
        for init in model.graph.initializer:
            if init.name == node.input[1]:
                # get the value as numpy array
                w = onnx.numpy_helper.to_array(init)
                print (w.shape)
                # make a new tensor with the same name, data type, dimensions and values
                
                a = random.randint(0, 255)
                
                value = onnx.helper.make_tensor(init.name + "_" + node.name, onnx.TensorProto.FLOAT, init.dims, a * w.flatten())
                # remove the old initializer and add the modified one
                # model.graph.initializer.remove(init)
                model.graph.initializer.append(value)
                # change the input of the node to point to the new initializer
                node.input[1] = init.name + "_" + node.name
                break
        npz_idx = npz_idx + 1
        print ("npz_logging/" + str(npz_idx) + "_mul.npz")
        val = np.load(npz_path + "/" + str(npz_idx) +  "_mul.npz")
        d = output_map[node.input[0]][0]
        attr = onnx.helper.make_attribute("A_bitdepth", 8)
        node.attribute.append(attr) 
        # attr = copy.deepcopy(get_attribute(d.attribute, "output_scale"))
        # a_scale = attr.f
        # attr.name = "A_scale"
        # node.attribute.append(attr) 
        
        attr = onnx.helper.make_attribute("A_scale", 1/val["input_A_scale"][0])
        node.attribute.append(attr) 
        a_scale=1/val["input_A_scale"][0]
        if node.input[0] in output_map:
            for d in output_map[node.input[0]]:
                if not has_attribute(d.attribute, "output_scale"):
                    attr = onnx.helper.make_attribute("output_bitdepth", 8)
                    d.attribute.append(attr)
                    attr = onnx.helper.make_attribute("output_scale", 1/val["input_A_scale"][0])
                    d.attribute.append(attr) 
        attr = onnx.helper.make_attribute("B_bitdepth", 8)
        node.attribute.append(attr) 
        # d = output_map[node.input[1]][0]
        # attr = copy.deepcopy(get_attribute(d.attribute, "output_scale"))
        # b_scale = attr.f
        # attr.name = "B_scale"
        # node.attribute.append(attr) 
        attr = onnx.helper.make_attribute("B_scale", 1/val["input_B_scale"][0])
        node.attribute.append(attr) 
        b_scale=1/val["input_B_scale"][0]
        if node.input[1] in output_map:
            for d in output_map[node.input[1]]:
                if not has_attribute(d.attribute, "output_scale"):
                    attr = onnx.helper.make_attribute("output_bitdepth", 8)
                    d.attribute.append(attr)
                    attr = onnx.helper.make_attribute("output_scale", 1/val["input_B_scale"][0])
                    d.attribute.append(attr) 
        attr = onnx.helper.make_attribute("output_bitdepth", 8)
        node.attribute.append(attr) 
        attr = onnx.helper.make_attribute("output_scale", 1/val["output_scale"][0])
        assert b_scale * a_scale >= 1/val["output_scale"][0], "scale" + str(b_scale) + " " + str(a_scale) + " " + str(1/val["output_scale"][0])
        
        print ("scale" + str(b_scale) + " " + str(a_scale) + " " + str(1/val["output_scale"][0]))
        
        node.attribute.append(attr) 
        
    elif node.op_type == "Resize":
        npz_idx = npz_idx + 1
        val = np.load(npz_path + "/" + str(npz_idx) + "_resize.npz")
        for input1 in node.input:
            if input1 in output_map:
                for d in output_map[input1]:
                    if not has_attribute(d.attribute, "output_scale"):
                        attr = onnx.helper.make_attribute("output_bitdepth", 8)
                        d.attribute.append(attr)
                        attr = onnx.helper.make_attribute("output_scale", 1/val["input_scale"][0])
                        d.attribute.append(attr) 
        attr = onnx.helper.make_attribute("input_bitdepth", 8)
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("input_scale", 1/val["input_scale"][0])
        node.attribute.append(attr) 
        attr = onnx.helper.make_attribute("output_bitdepth", 8)
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("output_scale", 1/val["input_scale"][0])
        node.attribute.append(attr) 
        
    # elif node.op_type == "Concat":
    #     npz_idx = npz_idx + 1
    #     val = np.load(npz_path + "/" + str(npz_idx) +  "_concat.npz")
    #     for input1 in node.input:
    #         if input1 in output_map:
    #             for d in output_map[input1]:
    #                 if not has_attribute(d.attribute, "output_scale"):
    #                     attr = onnx.helper.make_attribute("output_bitdepth", 8)
    #                     d.attribute.append(attr)
    #                     attr = onnx.helper.make_attribute("output_scale", 1/val["output_scale"][0])
    #                     d.attribute.append(attr) 
    #     attr = onnx.helper.make_attribute("output_bitdepth", 8)
    #     node.attribute.append(attr)
    #     attr = onnx.helper.make_attribute("output_scale", 1/val["output_scale"][0])
    #     node.attribute.append(attr) 
    
    elif node.op_type == "Concat":
        npz_idx = npz_idx + 1
        val = np.load(npz_path + "/" + str(npz_idx) + "_concat.npz")
        concat_output_scale = 1 / val["output_scale"][0]  # Define the authoritative scale
        
        # Force all input nodes to match the Concat node's output_scale
        for input1 in node.input:
            if input1 in output_map:
                for d in output_map[input1]:
                    # Remove any existing output_scale attribute
                    for attr in d.attribute[:]:
                        if attr.name == "output_scale":
                            d.attribute.remove(attr)
                    # Set the output_bitdepth and output_scale
                    if not has_attribute(d.attribute, "output_bitdepth"):
                        attr = onnx.helper.make_attribute("output_bitdepth", 8)
                        d.attribute.append(attr)
                    attr = onnx.helper.make_attribute("output_scale", concat_output_scale)
                    d.attribute.append(attr)
        
        # Set Concat node's attributes
        attr = onnx.helper.make_attribute("output_bitdepth", 8)
        node.attribute.append(attr)
        attr = onnx.helper.make_attribute("output_scale", concat_output_scale)
        node.attribute.append(attr)
        
    elif node.op_type == "Div":
        npz_idx = npz_idx + 1
    
    elif node.op_type == "ReduceSum":
        None
    else:
        d = output_map[node.input[0]][0]
        if has_attribute(d.attribute, "output_scale"):
            attr = copy.deepcopy(get_attribute(d.attribute, "output_scale"))
            node.attribute.append(attr) 
            attr = onnx.helper.make_attribute("output_bitdepth", 8)
            node.attribute.append(attr)    
        
attr = onnx.helper.make_attribute("output_bitdepth", 8)
node.attribute.append(attr)
attr = onnx.helper.make_attribute("output_scale", 4.0)
node.attribute.append(attr)      

name: "output_scale"
f: 4
type: FLOAT

##### Save the Final FXP ONNX

In [8]:
onnx.save( model, fxp_onnx_path)